# Riesz representer learners for the ATE and ATT: simulation study

This notebook compares two learners of the Riesz representer $\alpha$ when $\hat\alpha$ feeds a cross-fitted one-step estimator:

- **xgboost**: rieszboost, gradient boosting with XGBoost trees.
- **riesznet**: riesznet, a feed-forward neural network.

The write-up follows ADEMP (Morris, White & Crowther, 2019): **A**ims, **D**ata-generating mechanisms, **E**stimands, **M**ethods, and **P**erformance measures come first. The results follow, one section per aim.

**Where the code lives.** Each file covers one part of the design. Its module docstring states the formulas.

| File | Contents |
|---|---|
| `dgps.py` | The two DGPs: covariates, propensity score $\pi$, outcome regression $\mu$ |
| `estimands.py` | ATE and ATT: true $\alpha_0$, true $\psi_0$, one-step estimator and its standard error |
| `learners.py` | The outcome-regression learner, the two Riesz learners, their fixed settings and tuning grids |
| `simulation.py` | One simulated dataset end to end, the cache of fitted predictions, the loop over datasets |
| `summaries.py` | Every table and figure below, computed from the saved per-replicate results |
| `simulation.slurm` | The same run as a SLURM job |

**Running on a server.**

1. Clone the repo and keep this notebook in `examples/backend_comparison/`.
2. Install the dependencies into the kernel's environment, from the repo root:
   `pip install -e packages/rieszreg/python -e packages/rieszboost/python -e packages/riesznet/python xgboost torch scikit-learn pandas joblib scipy matplotlib`
3. Restart the kernel and run the cells top to bottom.

Every cross-fit is cached in `cache/` as soon as it finishes. If the kernel dies, rerun the notebook: it reloads finished fits and continues. Adding a learner or an estimator refits nothing that already exists.

## Setup

In [ ]:
# Limit each process to one thread. The simulated datasets already run in parallel,
# and torch and xgboost can crash on macOS when both use multithreaded OpenMP.
# This must run before numpy, torch, or xgboost is imported.
import os
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[var] = "1"

# Put this folder and the workspace packages on the path. PYTHONPATH is set too,
# so the parallel worker processes inherit it.
import sys
from pathlib import Path
HERE = Path.cwd()                                  # examples/backend_comparison
PATHS = [str(HERE)] + sorted(str(p) for p in HERE.parents[1].glob("packages/*/python"))
sys.path[:0] = PATHS
os.environ["PYTHONPATH"] = os.pathsep.join(PATHS + [os.environ.get("PYTHONPATH", "")])

import pandas as pd

from dgps import DGPS
from learners import OUTCOME_LEARNER, RIESZ_LEARNERS
from simulation import RESULTS_DIR, run_simulation, save
from summaries import (describe_dgp, early_stopping_check, edge_check, estimator_performance,
                       nuisance_accuracy, paired_contrasts, plot_errors, with_mcse)

## Settings

In [ ]:
RUN_DGPS = ["easy"]   # any of "easy", "hard"
N = 2000              # sample size n of each simulated dataset
N_SIM = 20            # replicates: 2 to check the code runs, 100-200 as a pilot, then the count from section R4
CV_FOLDS = 5          # GridSearchCV folds for tuning each learner
N_JOBS = 8            # datasets fit in parallel; at most the number of cores

## A. Aims

For the ATE and the ATT, at $n = 2000$, we ask:

1. **Representer accuracy.** Which learner estimates the true representer $\alpha_0$ more accurately, measured by the RMSE of $\hat\alpha$ against $\alpha_0$?
2. **Estimator performance.** Does that difference carry through to the one-step estimator? We measure the bias, empirical SE, RMSE, and 95% Wald interval coverage of $\hat\psi$, and whether the estimated SE matches the empirical SE.
3. **Dependence on the DGP.** Do the answers to 1 and 2 change between a small, nearly linear DGP and a large, nonlinear one?

Every performance measure in section P answers one of these aims. The claim the study supports is stated only after the full run, so the notebook does not assume which learner wins.

## D. Data-generating mechanisms

Both DGPs share this structure, with $p$ covariates:

$$
X \sim N(0, \Sigma),\ \ \Sigma_{jk} = 0.3^{|j-k|}, \qquad
A \mid X \sim \text{Bernoulli}(\pi(X)), \qquad
Y \mid A, X \sim N\big(\mu(A, X),\ 2.5^2\big),
$$

with $\pi(X)$ clipped to $[0.025, 0.975]$. The covariates fall into four blocks: confounders (affect $A$ and $Y$), outcome-only, treatment-only, and pure noise. The noise block tests whether a learner can ignore covariates that do not matter.

**Easy DGP** ($p = 10$). Confounders $x_1$–$x_4$, outcome-only $x_5, x_6$, treatment-only $x_7, x_8$, noise $x_9, x_{10}$.

$$
\begin{aligned}
\operatorname{logit} \pi(X) &= 0.1 + 0.5x_1 - 0.4x_2 + 0.3x_3 - 0.2x_4 + 0.25(x_1^2 - 1) + 0.3x_7 - 0.3x_8 \\
\mu(a, X) &= x_1 - 0.8x_2 + 0.6x_3 - 0.4x_4 + 0.5(x_2^2 - 1) + 0.8x_5 - 0.5x_6 + a\,(3 + 0.5x_1)
\end{aligned}
$$

**Hard DGP** ($p = 40$). Confounders $x_1$–$x_{20}$, outcome-only $x_{21}$–$x_{25}$, treatment-only $x_{26}$–$x_{30}$, noise $x_{31}$–$x_{40}$. Write $C = (x_1, \dots, x_{20})$ for the confounders. Then

$$
\begin{aligned}
\operatorname{logit} \pi(X) &= 0.1 + f(C; b_1) + g(C; b_2) + \textstyle\sum_{j=26}^{30} b_{3,j}\, x_j \\
\mu(a, X) &= f(C; c_1) + g(C; c_2) + \textstyle\sum_{j=21}^{25} c_{3,j}\, x_j + a\,\big(3 + f(C; c_4)\big),
\end{aligned}
$$

where $f(C; b) = \sum_j b_j h_j(C_j)$ and $h_j$ cycles through $x$, $x^2 - 1$, and $\sin(\pi x)$ as $j$ runs over the confounders. The pair term $g(C; b) = \sum_j b_j C_j C_{j+1}$ adds products of adjacent confounders. The treatment effect $3 + f(C; c_4)$ varies with all 20 confounders. Each coefficient vector alternates in sign and shrinks like $1/\sqrt{j}$, times a Uniform(0.7, 1.3) jitter. The coefficients are drawn once from a fixed seed, so the DGP is the same function in every replicate (`HardDGP` in `dgps.py`).

**Design.** One sample size and two DGPs. The DGPs differ in dimension and in nonlinearity at once. A difference between them therefore cannot be attributed to either factor alone.

**Diagnostics.** The table checks each DGP on one draw of 200,000 rows. It reports treatment prevalence, overlap, the share of $\text{Var}(Y)$ explained by $\mu$, and the share of $\text{Var}(\mu)$ captured by the best linear fit in $(A, X)$. It also reports the spread of the true representers, whose largest value is set by the propensity clip, and each $\psi_0$ with the Monte Carlo SE of its computation.

In [ ]:
pd.concat([describe_dgp(dgp) for dgp in DGPS.values()], axis=1).round(3)

## E. Estimands

Write $\tau(X) = \mu(1, X) - \mu(0, X)$.

- **ATE**: $\psi = \mathbb{E}[\tau(X)]$.
- **ATT**: $\psi = \mathbb{E}[\tau(X) \mid A = 1]$.

The true $\psi_0$ has no closed form in the hard DGP. For both DGPs we compute it by Monte Carlo over $2 \times 10^6$ draws of $X$, using the known $\mu$. For the ATT we weight $\tau(X)$ by $\pi(X)$ rather than drawing $A$. The table above lists the Monte Carlo SE of each $\psi_0$. It should be small next to the Monte Carlo SE of the bias in section R3.

## M. Methods

**Estimator.** Each replicate runs these steps (`run_cell` in `simulation.py`):

1. Draw $n$ observations $O = (Y, A, X)$. Replicate $r$ uses seed $510 + r$, so any replicate can be rerun on its own.
2. Split the rows into 2 cross-fitting folds.
3. For each fold, fit $\mu$ on the other fold. Predict $\hat\mu(A, X)$, $\hat\mu(1, X)$, and $\hat\mu(0, X)$ on this fold.
4. For each estimand and each Riesz learner, cross-fit $\hat\alpha$ over the same folds.
5. Plug $\hat\mu$ and $\hat\alpha$ into the one-step estimator.

Both Riesz learners share the same $\hat\mu$ and the same datasets. A difference in $\hat\psi$ between them therefore comes from $\hat\alpha$ alone, and every comparison is paired by replicate.

**One-step estimators** (`estimands.py`). For the ATE, $\alpha(A, X) = \dfrac{A}{\pi(X)} - \dfrac{1 - A}{1 - \pi(X)}$, and $\hat\psi$ and its SE are the mean and $\text{sd}/\sqrt{n}$ of

$$
\hat\varphi(O) = \hat\tau(X) + \hat\alpha(A, X)\,\big(Y - \hat\mu(A, X)\big).
$$

For the ATT, write $\psi = \psi^* / P(A = 1)$ with $\psi^* = \mathbb{E}[A\,\tau(X)]$. The learners fit the representer of $\psi^*$, $\alpha(A, X) = A - (1 - A)\,\dfrac{\pi(X)}{1 - \pi(X)}$. With $\hat p$ the sample mean of $A$,

$$
\hat\psi = \frac{1}{\hat p} \cdot \frac{1}{n}\sum_{i=1}^n \Big[A_i\,\hat\tau(X_i) + \hat\alpha(A_i, X_i)\big(Y_i - \hat\mu(A_i, X_i)\big)\Big],
\qquad
\hat\varphi(O) = \frac{A\,\big(\hat\tau(X) - \hat\psi\big) + \hat\alpha(A, X)\big(Y - \hat\mu(A, X)\big)}{\hat p},
$$

and the SE is $\text{sd}(\hat\varphi)/\sqrt{n}$. The 95% interval is $\hat\psi \pm 1.96\,\text{SE}$.

**Learners** (`learners.py`). Each hyperparameter is either fixed or tuned.

| Nuisance | Learner | Fixed | Tuned |
|---|---|---|---|
| $\mu$ | XGBoost regression | up to 10,000 trees; early stopping, patience 20, on a held-out 20% | depth, learning rate, L2 penalty |
| $\alpha$ | **xgboost** (rieszboost) | up to 20,000 trees; early stopping, patience 20, on a held-out 20% of individuals; each round fits on 80% of individuals | depth, learning rate |
| $\alpha$ | **riesznet** | as in Lee & Schuler: 3 hidden layers of width 200, ELU, AdamW with weight decay $10^{-3}$, up to 1,000 epochs, early stopping, patience 10, on a held-out 20%; keeps the better of 2 initializations | learning rate |

The tree and epoch counts are caps. Early stopping chooses how many to keep, and section R1 checks that the caps do not bind. Every fit uses one thread and a fixed seed.

**Tuning.** Inside each cross-fitting training fold, `GridSearchCV` with `CV_FOLDS` folds scores every grid point with the learner's own `score` method: $R^2$ for the outcome regression, and the negative held-out squared Riesz loss $-\frac{1}{n}\sum_j [\hat\alpha(Z_j)^2 - 2\,m(\hat\alpha)(Z_j)]$ for the Riesz learners. It then refits the best point on the whole training fold. The grids:

In [ ]:
grids = {OUTCOME_LEARNER.name: OUTCOME_LEARNER.grid} | {name: l.grid for name, l in RIESZ_LEARNERS.items()}
pd.DataFrame(
    [(learner, param.removeprefix("estimator__"), values) for learner, grid in grids.items() for param, values in grid.items()],
    columns=["learner", "hyperparameter", "grid"],
)

## P. Performance measures

Over $n_{\text{sim}}$ replicates with estimates $\hat\psi_i$ and SEs $\widehat{\text{SE}}_i$. Every measure is reported with its Monte Carlo SE (MCSE).

| Measure | Definition | MCSE | Aim |
|---|---|---|---|
| RMSE of $\hat\alpha$ | $\big(\frac{1}{n}\sum_j (\hat\alpha(Z_j) - \alpha_0(Z_j))^2\big)^{1/2}$ within a replicate, averaged over replicates | $\text{sd}/\sqrt{n_{\text{sim}}}$ | 1 |
| Bias | $\frac{1}{n_{\text{sim}}}\sum_i \hat\psi_i - \psi_0$ | $\text{EmpSE}/\sqrt{n_{\text{sim}}}$ | 2 |
| Empirical SE | $\text{sd}(\hat\psi_i)$ | $\text{EmpSE}/\sqrt{2(n_{\text{sim}} - 1)}$ | 2 |
| Model SE | $\big(\frac{1}{n_{\text{sim}}}\sum_i \widehat{\text{SE}}_i^2\big)^{1/2}$, compared with the empirical SE | delta method | 2 |
| RMSE of $\hat\psi$ | $\big(\frac{1}{n_{\text{sim}}}\sum_i (\hat\psi_i - \psi_0)^2\big)^{1/2}$ | delta method from the MCSE of the MSE | 2 |
| Coverage | share of intervals $\hat\psi_i \pm 1.96\,\widehat{\text{SE}}_i$ that contain $\psi_0$ | $\sqrt{\text{cov}(1 - \text{cov})/n_{\text{sim}}}$ | 2 |

Aim 3 compares these measures across the DGP columns. The number of failed fits is reported first, next to $n_{\text{sim}}$.

**Number of replicates.** Section R4 contrasts the two learners replicate by replicate. It estimates how many replicates are needed to show each gap at 5 MCSEs. Run a pilot of 100 to 200 replicates, then set `N_SIM` from that section.

## Run

Reruns reload every finished fit from `cache/`, so this cell is cheap after the first time. The per-replicate tables are saved to `results/` as the record behind every display.

In [ ]:
tables = run_simulation(RUN_DGPS, [N], range(N_SIM), cv_folds=CV_FOLDS, n_jobs=N_JOBS)
save(tables, RESULTS_DIR / f"{'_'.join(RUN_DGPS)}_n{N}")
results, tuning = tables["results"], tables["tuning"]

## Results

### R1. Are the learners tuned well enough to compare?

A comparison of learners is only fair if each is tuned near its best. Otherwise it compares tuning grids. Two checks come from the stored CV score of every grid point.

**Edge check.** For each tuned hyperparameter, the share of fits whose selected value is the smallest or largest in its grid. Occasional edge choices are noise. If one edge is chosen consistently, the best setting may lie beyond the grid. The grid should then shift in that direction, dropping points at the other end so the runtime stays flat. A two-point grid has no interior, so every choice is an edge, and the check cannot pass for it.

In [ ]:
edge_check(tuning).round(2)

**Early stopping.** The kept iteration relative to its cap. If fits keep reaching 90% of the cap, the learner wants more trees or epochs. Raise the learning rate rather than the cap.

In [ ]:
early_stopping_check(tuning).round(1)

The replicates with the largest error in $\hat\alpha$. Runaway fits show up here first.

In [ ]:
(results.assign(error=results.est - results.truth)
        .nlargest(5, "alpha_rmse")[["dgp", "rep", "estimand", "learner", "est", "truth", "error", "se", "alpha_rmse"]]
        .round(3))

### R2. Aim 1: which learner estimates $\alpha_0$ more accurately?

Mean RMSE and MAE of $\hat\alpha$ against $\alpha_0$, with MCSEs in parentheses. The RMSE of $\hat\mu$ is the same for both learners because they share it. Read the two learners' rows side by side within each estimand. Section R4 tests whether the gap exceeds Monte Carlo error.

In [ ]:
with_mcse(nuisance_accuracy(results))

### R3. Aim 2: does the difference carry through to $\hat\psi$?

Performance of the one-step estimator, with MCSEs in parentheses. A bias within 2 MCSEs of zero is indistinguishable from zero at this $n_{\text{sim}}$. When coverage falls short of 95%, check in order: bias relative to the empirical SE, then the model SE against the empirical SE. A model SE below the empirical SE means the intervals are too narrow.

In [ ]:
with_mcse(estimator_performance(results))

The table hides skew and outliers, so the figure shows $\hat\psi - \psi_0$ in every replicate. Grey lines join the two learners' estimates on the same dataset. Parallel lines mean the learners agree replicate by replicate. The black bar is the bias $\pm 1.96$ MCSE.

In [ ]:
plot_errors(results);

### R4. Are the differences between learners real, and how many replicates are needed?

Each row is a contrast, xgboost minus riesznet, paired by replicate: the mean per-replicate difference (`gap`), its MCSE, and $z = |\text{gap}|/\text{MCSE}$. When $z < 2$ this run cannot tell the gap from zero. That can mean the learners agree, or that $n_{\text{sim}}$ is too small, and this table alone cannot say which. `n_sim_needed` is the number of replicates that would show the gap at 5 MCSEs, sized on the lower 95% bound of the gap. It is only meaningful when $z \ge 2$. Use the largest value across the contrasts the paper reports to set `N_SIM`.

In [ ]:
paired_contrasts(results).round(4)